# DSS — you get what you ask for

DSS finds spatial filters that maximise a property you *declare*, rather than variance. Declaring the right property is the whole job.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

## Same data, two different biases

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_ep, n_ch, n_t, sfreq = 120, 32, 256, 256.0
times = np.arange(n_t) / sfreq - 0.2

# an evoked response that repeats across trials
evoked = -np.exp(-((times - 0.15) ** 2) / 0.002)
v_evoked = rng.standard_normal(n_ch); v_evoked /= np.linalg.norm(v_evoked)

# a 10 Hz rhythm with random phase per trial (NOT phase-locked)
v_alpha = rng.standard_normal(n_ch); v_alpha /= np.linalg.norm(v_alpha)

data = rng.standard_normal((n_ep, n_ch, n_t)) * 0.7
for e in range(n_ep):
    data[e] += np.outer(v_evoked, evoked) * 1.1
    data[e] += np.outer(v_alpha, np.sin(2 * np.pi * 10 * times + rng.uniform(0, 6.28))) * 1.4

info = mne.create_info(n_ch, sfreq, "eeg")
epochs = mne.EpochsArray(data, info, tmin=-0.2, verbose="ERROR")
print(f"{n_ep} trials: a phase-locked evoked response AND a stronger non-phase-locked 10 Hz rhythm")

In [ ]:
from mne_denoise.dss import DSS, AverageBias, BandpassBias

dss_avg = DSS(bias=AverageBias(axis="epochs"), n_components=4)
src_avg = dss_avg.fit_transform(epochs)

dss_band = DSS(bias=BandpassBias(freq_band=(8.0, 12.0), sfreq=sfreq), n_components=4)
src_band = dss_band.fit_transform(epochs)

print("AverageBias  eigenvalues:", np.round(dss_avg.eigenvalues_[:4], 3))
print("BandpassBias eigenvalues:", np.round(dss_band.eigenvalues_[:4], 3))
print("\nsimilarity of component 1 to each planted pattern:")
for label, d in [("AverageBias", dss_avg), ("BandpassBias", dss_band)]:
    p = d.patterns_[:, 0]; p = p / np.linalg.norm(p)
    print(f"  {label:14s} evoked {abs(p @ v_evoked):.2f}   alpha {abs(p @ v_alpha):.2f}")

The alpha rhythm has more variance. PCA would return it. `AverageBias` returns the evoked response instead — because that is what was asked for.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, src, title, colour in [
    (axes[0], src_avg, "AverageBias — component 1", "#0072B2"),
    (axes[1], src_band, "BandpassBias 8-12 Hz — component 1", "#009E73"),
]:
    for tr in np.asarray(src)[::4, 0, :]:
        ax.plot(times, tr, color=colour, alpha=0.06, lw=0.8)
    ax.plot(times, np.asarray(src)[:, 0, :].mean(0), color=colour, lw=3)
    ax.axvline(0, color="#888888", ls=":", lw=1)
    ax.set_title(title); ax.set_xlabel("Time (s)")
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

> **And the caution.** On real ERP CORE N170 data, trial-average DSS improved split-half reproducibility in 37 of 40 participants but held-out faces-vs-cars discriminability in only 25 of 40. Concentrating what repeats across trials is not the same as sharpening a condition contrast.